In [80]:
import heapq
import csv 
from collections import deque
import glob
import logging
import math
import os
import shutil
import warnings

import gymnasium as gym
import numpy as np
import pandas as pd
from pandas.tseries.offsets import BDay
import ray
from ray import air, tune
from ray.rllib.algorithms.algorithm import Algorithm
from ray.rllib.algorithms.callbacks import DefaultCallbacks
from ray.tune.registry import get_trainable_cls

warnings.filterwarnings("ignore")
pd.set_option('display.max_rows', 900)

### load market state data

In [81]:
df = pd.read_csv('./ASML.csv', index_col='timestamp').astype(np.float32)
df.head(3)

,open,high,low,close,volume,EMA3,EMA5,EMA10,EMA15,RSI,MACD,MACD_signal,MACD_hist,MFI,DX
timestamp,,,,,,,,,,,,,,,
2016-11-25 10:05:00-05:00,103.959999,103.980003,103.940002,103.940002,436.0,103.957565,103.965515,103.980972,103.991425,45.438419,-0.019043,-0.010334,-0.008709,27.485161,21.583424
2016-11-25 10:06:00-05:00,103.980003,103.980003,103.970001,103.980003,1400.0,103.968781,103.970345,103.980797,103.989998,49.786537,-0.018158,-0.011899,-0.006259,30.894516,21.583424
2016-11-25 10:07:00-05:00,104.010002,104.010002,104.010002,104.010002,230.0,103.989388,103.983559,103.986107,103.992493,52.823147,-0.014865,-0.012492,-0.002373,31.588350,12.280625


In [82]:
with pd.option_context('display.float_format', '{:.5f}'.format):
    display(df.describe())

,open,high,low,close,volume,EMA3,EMA5,EMA10,EMA15,RSI,MACD,MACD_signal,MACD_hist,MFI,DX
count,635671.00000,635671.00000,635671.00000,635671.00000,635671.00000,635671.00000,635671.00000,635671.00000,635671.00000,635671.00000,635671.00000,635671.00000,635671.00000,635671.00000,635671.00000
mean,406.66794,406.79572,406.53775,406.66586,2563.06299,406.66498,406.66412,406.66177,406.65942,50.48303,0.00647,0.00647,-0.00000,49.82185,22.37874
std,224.03951,224.11163,223.96533,224.03773,9807.82422,224.03722,224.03687,224.03606,224.03526,11.71732,0.63430,0.59724,0.19108,20.12563,16.61563
min,98.65000,98.65000,98.65000,98.65000,3.00000,98.81013,98.83862,98.90675,98.98062,2.47228,-13.92212,-11.66004,-5.49650,-0.00000,0.00000
25%,194.16000,194.22000,194.10001,194.17000,509.00000,194.15961,194.16208,194.16348,194.16162,42.79423,-0.15160,-0.14410,-0.04860,35.37834,9.03471
50%,367.73001,367.85999,367.60001,367.73001,1158.00000,367.71249,367.69797,367.69412,367.69995,50.56857,0.00903,0.00899,-0.00051,49.72668,19.07353
75%,621.50000,621.71997,621.26001,621.46997,2440.00000,621.46451,621.47232,621.49326,621.49289,58.23144,0.17274,0.16474,0.04716,64.18087,32.42533
max,902.71997,902.71997,902.71997,902.71997,2302216.00000,902.09967,901.25122,898.76569,896.96130,97.21130,8.60534,7.95226,3.51867,100.00000,95.80043


### integrate sentiment scores

In [83]:
df_news = pd.read_csv('./ASML_news.csv')
df_news['sentiment_score'] = pd.read_csv('./sentiment_scores.csv', header=None).iloc[0].values.astype(np.float32)
df_news['timestamp_rounded'] = pd.to_datetime(df_news['timestamp']).dt.round('T')
df_news = df_news.drop(['text', 'timestamp'], axis=1).set_index('timestamp_rounded')
df_news = df_news[~df_news.index.duplicated()].resample('T').ffill() # expand to consecutive minutes and forward-fill sentiment scores
df.index = pd.to_datetime(df.index)
df = pd.merge(df, df_news, left_index=True, right_index=True, how='left')
df['sentiment_score'].fillna(0, inplace=True)
df.head()

,open,high,low,close,volume,EMA3,EMA5,EMA10,EMA15,RSI,MACD,MACD_signal,MACD_hist,MFI,DX,sentiment_score
timestamp,,,,,,,,,,,,,,,,
2016-11-25 10:05:00-05:00,103.959999,103.980003,103.940002,103.940002,436.0,103.957565,103.965515,103.980972,103.991425,45.438419,-0.019043,-0.010334,-0.008709,27.485161,21.583424,0.0
2016-11-25 10:06:00-05:00,103.980003,103.980003,103.970001,103.980003,1400.0,103.968781,103.970345,103.980797,103.989998,49.786537,-0.018158,-0.011899,-0.006259,30.894516,21.583424,0.0
2016-11-25 10:07:00-05:00,104.010002,104.010002,104.010002,104.010002,230.0,103.989388,103.983559,103.986107,103.992493,52.823147,-0.014865,-0.012492,-0.002373,31.588350,12.280625,0.0
2016-11-25 10:08:00-05:00,103.970001,103.970001,103.970001,103.970001,100.0,103.979698,103.979042,103.983177,103.989685,48.602749,-0.015307,-0.013055,-0.002252,80.294563,20.963984,0.0
2016-11-25 10:09:00-05:00,103.949997,103.970001,103.949997,103.949997,517.0,103.964851,103.969360,103.977142,103.984726,46.598042,-0.017074,-0.013859,-0.003215,77.707077,24.963604,0.0


In [84]:
df['sentiment_score'].value_counts()

sentiment_score
 0.0    351898
 1.0    208724
-1.0     75049
Name: count, dtype: int64

### transform/normalize market state data

In [85]:
market_state_data = df.copy()

market_state_data['volume'] = market_state_data['volume'].apply(lambda x: x * 2 ** -10).astype(np.float32)

pricing_col_names = ['open', 'high', 'low', 'close']
for col_name in pricing_col_names:
    market_state_data[col_name] = market_state_data[col_name].apply(lambda x: x * 2 ** -6).astype(np.float32)

technical_indicator_col_names = ['EMA3', 'EMA5', 'EMA10', 'EMA15']
for col_name in technical_indicator_col_names:
    market_state_data[col_name] = market_state_data[col_name].apply(lambda x: x * 2 ** -7).astype(np.float32)

technical_indicator_col_names = ['RSI', 'MFI', 'DX']
for col_name in technical_indicator_col_names:
    market_state_data[col_name] = market_state_data[col_name].apply(lambda x: x * 2 ** -3).astype(np.float32)

split_index = int(market_state_data.shape[0]*0.8)
train_market_state_data = market_state_data.iloc[:split_index]
test_market_state_data = market_state_data.iloc[split_index:]
display(train_market_state_data.head(3))
display(test_market_state_data.head(3))
display(train_market_state_data.shape, test_market_state_data.shape)
num_trading_days = pd.date_range(start=test_market_state_data.index[0], end=test_market_state_data.index[-1], freq=BDay()).shape[0] # count number of business days (trading days)
print(f'Number of trading days in test data: {num_trading_days}')

,open,high,low,close,volume,EMA3,EMA5,EMA10,EMA15,RSI,MACD,MACD_signal,MACD_hist,MFI,DX,sentiment_score
timestamp,,,,,,,,,,,,,,,,
2016-11-25 10:05:00-05:00,1.624375,1.624688,1.624063,1.624063,0.425781,0.812168,0.812231,0.812351,0.812433,5.679802,-0.019043,-0.010334,-0.008709,3.435645,2.697928,0.0
2016-11-25 10:06:00-05:00,1.624688,1.624688,1.624531,1.624688,1.367188,0.812256,0.812268,0.812350,0.812422,6.223317,-0.018158,-0.011899,-0.006259,3.861814,2.697928,0.0
2016-11-25 10:07:00-05:00,1.625156,1.625156,1.625156,1.625156,0.224609,0.812417,0.812372,0.812391,0.812441,6.602893,-0.014865,-0.012492,-0.002373,3.948544,1.535078,0.0


,open,high,low,close,volume,EMA3,EMA5,EMA10,EMA15,RSI,MACD,MACD_signal,MACD_hist,MFI,DX,sentiment_score
timestamp,,,,,,,,,,,,,,,,
2022-08-08 10:04:00-05:00,8.957344,8.962500,8.952344,8.961329,3.293945,4.479555,4.479210,4.479140,4.479871,5.970599,-0.366496,-0.452447,0.085951,5.160603,0.164767,0.0
2022-08-08 10:05:00-05:00,8.960156,8.960156,8.960156,8.960156,0.442383,4.479816,4.479499,4.479311,4.479897,5.885628,-0.332169,-0.428391,0.096222,4.944951,0.164767,0.0
2022-08-08 10:06:00-05:00,8.965313,8.969375,8.965313,8.969375,0.881836,4.482252,4.481229,4.480288,4.480495,6.597300,-0.254424,-0.393598,0.139174,4.885647,1.348697,0.0


(508536, 16)

(127135, 16)

Number of trading days in test data: 340


In [86]:
with pd.option_context('display.float_format', '{:.5f}'.format):
    display(market_state_data.describe())

,open,high,low,close,volume,EMA3,EMA5,EMA10,EMA15,RSI,MACD,MACD_signal,MACD_hist,MFI,DX,sentiment_score
count,635671.00000,635671.00000,635671.00000,635671.00000,635671.00000,635671.00000,635671.00000,635671.00000,635671.00000,635671.00000,635671.00000,635671.00000,635671.00000,635671.00000,635671.00000,635671.00000
mean,6.35419,6.35618,6.35215,6.35415,2.50299,3.17707,3.17706,3.17705,3.17703,6.31038,0.00647,0.00647,-0.00000,6.22773,2.79734,0.21029
std,3.50062,3.50174,3.49946,3.50059,9.57795,1.75029,1.75029,1.75028,1.75028,1.46466,0.63430,0.59724,0.19108,2.51570,2.07695,0.63419
min,1.54141,1.54141,1.54141,1.54141,0.00293,0.77195,0.77218,0.77271,0.77329,0.30904,-13.92212,-11.66004,-5.49650,-0.00000,0.00000,-1.00000
25%,3.03375,3.03469,3.03281,3.03391,0.49707,1.51687,1.51689,1.51690,1.51689,5.34928,-0.15160,-0.14410,-0.04860,4.42229,1.12934,0.00000
50%,5.74578,5.74781,5.74375,5.74578,1.13086,2.87275,2.87264,2.87261,2.87266,6.32107,0.00903,0.00899,-0.00051,6.21583,2.38419,0.00000
75%,9.71094,9.71437,9.70719,9.71047,2.38281,4.85519,4.85525,4.85542,4.85541,7.27893,0.17274,0.16474,0.04716,8.02261,4.05317,1.00000
max,14.10500,14.10500,14.10500,14.10500,2248.25781,7.04765,7.04103,7.02161,7.00751,12.15141,8.60534,7.95226,3.51867,12.50000,11.97505,1.00000


In [123]:
# https://github.com/ray-project/ray/blob/master/rllib/examples/custom_metrics_and_callbacks.py
# https://docs.ray.io/en/latest/_modules/ray/rllib/algorithms/callbacks.html
class CustomCallbacks(DefaultCallbacks):
    def __init__(self):
        self.episode_index = 0
        if not os.path.exists('logs'):
            os.makedirs('logs', exist_ok=True)

    def on_episode_start(self, *, worker, base_env, policies, episode, env_index, **kwargs):
        env = worker.env
        csv_file_path = f'logs/{env.model_name}_worker_{worker.worker_index}_episode_{self.episode_index}.csv'
        log_file_path = f'logs/{env.model_name}_worker_{worker.worker_index}_episode_{self.episode_index}.log'
        self.file = open(csv_file_path, 'w', newline='')
        self.writer = csv.writer(self.file)
        header = ['time_index', 'timestamp', 'portfolio_value', 'cash_balance', 'profit', 'asset_price', 'order_size', 'borrow_size', 'leverage_size', 'margin_interest', 'asset_position']
        self.writer.writerow(header)
        logging.basicConfig(filename=log_file_path, format='%(message)s', level=logging.INFO)
        self.logger = logging.getLogger(f'{env.model_name}_worker_{worker.worker_index}_log') # log to the same file for human reader for now rather than saving to different files as the .csv files indexed by episode
        self.logger.info("time_index | portfolio_value |  cash_balance  |    profit    | asset_price | order_size | borrow_size | leverage_size | margin_interest | asset_position")
        self.logger.info("-----------|-----------------|----------------|--------------|-------------|------------|-------------|---------------|-----------------|---------------")
        metrics = {'time_index': env.time_index,
                   'timestamp': env.market_state_data.index[env.time_index],
                   'portfolio_value': round(env.portfolio_value, 2),
                   'cash_balance': round(env.initial_balance, 2),
                   'profit': 0,
                   'asset_price': round(env.market_state_data['close'].iloc[env.time_index]/env.pricing_scale, 2),
                   'order_size': 0,
                   'borrow_size': 0,
                   'leverage_size': 0,
                   'margin_interest': 0,
                   'asset_position': env.asset_position
                   }
        self.writer.writerow(metrics.values())
        self.logger.info(
            f"{metrics['time_index']:>10} | "
            f"{metrics['portfolio_value']:>15,.2f} | "
            f"{metrics['cash_balance']:>14,.2f} | "
            f"{metrics['profit']:>12,.2f} | "
            f"{metrics['asset_price']:>11.2f} | "
            f"{metrics['order_size']:>10} | "
            f"{metrics['borrow_size']:>11} | "
            f"{metrics['leverage_size']:>13} | "
            f"{metrics['margin_interest']:>15,.3f} | "
            f"{metrics['asset_position']:>14}"
        )
    def on_episode_step(self, *, worker, base_env, policies, episode, env_index, **kwargs):
        metrics = episode.last_info_for()
        self.writer.writerow(metrics.values())
        self.logger.info(
            f"{metrics['time_index']:>10} | "
            f"{metrics['portfolio_value']:>15,.2f} | "
            f"{metrics['cash_balance']:>14,.2f} | "
            f"{metrics['profit']:>12,.2f} | "
            f"{metrics['asset_price']:>11.2f} | "
            f"{metrics['order_size']:>10} | "
            f"{metrics['borrow_size']:>11} | "
            f"{metrics['leverage_size']:>13} | "
            f"{metrics['margin_interest']:>15,.3f} | "
            f"{metrics['asset_position']:>14}"
        )
    def on_episode_end(self, *, worker, base_env, policies, episode, env_index, **kwargs):
        self.episode_index += 1
        self.file.close()

# https://gymnasium.farama.org/api/env/
class CustomEnv(gym.Env):
    def __init__(self, env_config):
        self.model_name = env_config['model_name']
        self.initial_balance = env_config['initial_balance']
        self.market_state_data = env_config['market_state_data']
        self.reward_option = env_config['reward_option']
        self.allow_leverage = env_config['allow_leverage']
        self.recent_order_sizes = deque(maxlen=350)
        self.recent_asset_positions = deque(maxlen=350) # on the order of 60*6.5=390 trading minutes per trading day
        self.transaction_cost_rate = 1e-3
        self.pricing_scale = 2**-7
        self.asset_position_scale = 1/50
        self.cash_balance_scale = 1e-5
        self.leverage_size_scale = 1e-2
        self.reward_scale = 1
        self.order_size_threshold = 3
        self.long_initial_margin = 0.25
        self.long_maintenance_margin = 0.5
        self.short_initial_margin = 0.3
        self.short_maintenance_margin = 0.5
        self.leverage_maintenance_requirements = []
        self.margin_call_liquidation = False
        self.margin_interest_rate = 0.0000001464 # convert a 8% annual interest rate to a per-minute rate
        self.time_index = 0 # time index for asset prices
        self.cash_balance = self.initial_balance
        self.asset_position = 0
        self.portfolio_value = self.initial_balance
        self.max_episode_steps = self.market_state_data.shape[0] - 1
        self.action_dim = 1 # one asset to operate on for now
        self.state_dim = self.market_state_data.shape[1] + 4 # 4 additional dimensions for portfolio_value, cash_balance, asset_position, and leverage size
        self.action_space = gym.spaces.Box(low=-50, high=50, shape=(self.action_dim,), dtype=np.float32)
        self.observation_space = gym.spaces.Box(low=-np.infty, high=np.infty, shape=(self.state_dim,), dtype=np.float32)

    def reset(self, *, seed=None, options=None):
        # reset the environment to the initial state
        self.time_index = 0
        self.recent_order_sizes.clear()
        self.leverage_maintenance_requirements.clear()
        self.margin_call_liquidation = False
        self.cash_balance = self.initial_balance
        self.asset_position = 0
        self.portfolio_value = self.cash_balance
        state = np.hstack((self.market_state_data.iloc[self.time_index].values, 
                           self.portfolio_value*self.cash_balance_scale,
                           self.cash_balance*self.cash_balance_scale,
                           self.asset_position*self.asset_position_scale,
                           sum(leverage[3] for leverage in self.leverage_maintenance_requirements)*self.leverage_size_scale))
        return state, {} # state is market state combined with portfolio state
    
    def consecutive_inaction(self):
        max_num_consecutive_inaction = 0
        current_num_consecutive_inaction = 0
        for order_size in self.recent_order_sizes:
            if order_size == 0:
                current_num_consecutive_inaction += 1
                max_num_consecutive_inaction = max(max_num_consecutive_inaction, current_num_consecutive_inaction)
            else:
                current_num_consecutive_inaction = 0
        return max_num_consecutive_inaction
    
    def order_inbalance(self):
        num_long_orders = sum([order_size for order_size in self.recent_order_sizes if order_size > 0])
        num_short_orders = -sum([order_size for order_size in self.recent_order_sizes if order_size < 0])
        return abs(num_long_orders - num_short_orders)

    def step(self, action):
        # interact with the environment, compute reward, and update state
        self.time_index += 1
        order_size = int(action[0])
        asset_price = self.market_state_data['close'].iloc[self.time_index]/self.pricing_scale # recover the actual asset price
        
        # update the portfolio state accordingly based on order size
        borrow_size = 0
        if self.allow_leverage:
            # check whether the portfolio complies the most urging leverage maintenance requirements and update the portfolio state accordingly
            if len(self.leverage_maintenance_requirements) > 0:
                leverage_size = sum(leverage[3] for leverage in self.leverage_maintenance_requirements)
                leverage_maintenance_requirement = sum(leverage[0] for leverage in self.leverage_maintenance_requirements) # a value could be either negative or positive
                if leverage_size < 0 and (self.portfolio_value < abs(leverage_maintenance_requirement) or self.cash_balance < abs(leverage_maintenance_requirement)): # short positions cannot be kept any more
                    force_liquidation_size = abs(sum(leverage[3] for leverage in self.leverage_maintenance_requirements)) # number of assets must be returned to the broker; for simplicity, forced liquidation size is all leverage size
                    self.asset_position += force_liquidation_size # cover short positions and return all assets bought back to the broker
                    self.cash_balance -= force_liquidation_size*asset_price*(1 - self.transaction_cost_rate) # cash balance after forced liquidation
                    self.leverage_maintenance_requirements.clear()
                    if self.cash_balance < 0: # if cash cannot cover forced liquidation
                        if self.asset_position > 0:
                            self.cash_balance += self.asset_position*asset_price*(1 - self.transaction_cost_rate) # liquidate all the positions for cash
                            self.asset_position -= self.asset_position
                            if self.cash_balance < 0:
                                self.margin_call_liquidation = True # additional money owed to the broker
                        else:
                            self.margin_call_liquidation = True # additional money owed to the broker
                if leverage_size > 0 and (self.portfolio_value < abs(leverage_maintenance_requirement) or self.cash_balance < abs(leverage_maintenance_requirement)): # long positions cannot be kept any more
                    force_liquidation_size = abs(sum(leverage[3] for leverage in self.leverage_maintenance_requirements)) # for simplicity, forced liquidation size is all leverage size
                    borrowed_amount = abs(sum(leverage[2]*leverage[3] for leverage in self.leverage_maintenance_requirements)) # amount borrowed from broker for leveraged orders
                    self.asset_position -= force_liquidation_size
                    self.cash_balance += force_liquidation_size*asset_price*(1 - self.transaction_cost_rate) - borrowed_amount # cash balance after forced liquidation and returning borrowed capital to the broker
                    self.leverage_maintenance_requirements.clear()
                    if self.cash_balance < 0: # if cash cannot cover forced liquidation
                        if self.asset_position > 0:
                            self.cash_balance += self.asset_position*asset_price*(1 - self.transaction_cost_rate) # liquidate all the positions for cash
                            self.asset_position -= self.asset_position
                            if self.cash_balance < 0:
                                self.margin_call_liquidation = True # additional money owed to the broker
                        else:
                            self.margin_call_liquidation = True # additional money owed to the broker

            if order_size < 0 and order_size + self.order_size_threshold < 0: # short order
                if self.asset_position + order_size < 0:
                    borrow_size = self.asset_position + order_size if self.asset_position > 0 else order_size # a negative number
                    expected_leverage_size = sum(leverage[3] for leverage in self.leverage_maintenance_requirements) + borrow_size
                    margin_requirement = abs(expected_leverage_size)*asset_price*(1 + self.short_initial_margin) # e.g. shorting $100 worth of assets with leverage requires $130 initial capital
                    if self.portfolio_value > margin_requirement and self.cash_balance > margin_requirement: # non-collateral leverage, i.e., both portfolio value and cash balance must satisfy margin requirement
                        leverage_maintenance_requirement = -borrow_size*asset_price*(1 + self.short_maintenance_margin) # a positive number
                        margin_interest = -borrow_size*asset_price*self.margin_interest_rate # a positive number
                        heapq.heappush(self.leverage_maintenance_requirements, [-leverage_maintenance_requirement, # heapq sorts the minimum based on the first element of the tuple
                                                                                self.time_index, 
                                                                                asset_price,
                                                                                borrow_size,
                                                                                margin_interest,
                                                                                'short'])
                    else: # if margin requirement cannot be met
                        order_size = -self.asset_position
            elif order_size > 0 and order_size - self.order_size_threshold > 0: # long order
                if order_size > int(self.cash_balance//asset_price):
                    borrow_size = order_size - int(self.cash_balance//asset_price)
                    self.cash_balance += borrow_size*asset_price*(1 + self.transaction_cost_rate) # broker adds necessary cash balance so that leveraged order can be placed
                    expected_leverage_size = sum(leverage[3] for leverage in self.leverage_maintenance_requirements) + borrow_size
                    margin_requirement = expected_leverage_size*asset_price*self.long_initial_margin # e.g. longing $100 worth of assets with leverage requires $25 initial capital
                    if self.portfolio_value > margin_requirement and self.cash_balance > margin_requirement: # non-collateral leverage, i.e., both portfolio value and cash balance must satisfy margin requirement
                        leverage_maintenance_requirement = order_size*asset_price*self.long_maintenance_margin + self.cash_balance # non-collateral (based on cash balance) leverage
                        margin_interest = (order_size*asset_price - self.cash_balance)*self.margin_interest_rate
                        heapq.heappush(self.leverage_maintenance_requirements, [-leverage_maintenance_requirement, # heapq sorts the minimum based on the first element of the tuple
                                                                                self.time_index, 
                                                                                asset_price,
                                                                                borrow_size,
                                                                                margin_interest,
                                                                                'long'])
                    else: # if margin requirement cannot be met
                        order_size = int(self.cash_balance//asset_price)
            else:
                order_size = 0
            # cover short positions if there's any
            if borrow_size == 0 and order_size > 0 and len(self.leverage_maintenance_requirements) > 0:
                additional_long_size = self.leverage_maintenance_requirements[0][3] + order_size
                while additional_long_size > 0:
                    heapq.heappop(self.leverage_maintenance_requirements)
                    if len(self.leverage_maintenance_requirements) > 0:
                        additional_long_size = self.leverage_maintenance_requirements[0][3] + additional_long_size
                    else:
                        break
                else:
                    self.leverage_maintenance_requirements[0][3] = additional_long_size # remained borrow size (negative)
                    self.leverage_maintenance_requirements[0][4] = additional_long_size*self.leverage_maintenance_requirements[0][2]*self.margin_interest_rate
            
        else:
            if order_size < 0 and order_size + self.order_size_threshold < 0: # short order (no shorting)
                order_size = -min(self.asset_position, -order_size)
            elif order_size > 0 and order_size - self.order_size_threshold > 0: # long order (no leveraging)
                order_size = min(int(self.cash_balance//asset_price), order_size)
            else:
                order_size = 0

        self.recent_order_sizes.append(order_size)
        self.asset_position += order_size
        self.recent_asset_positions.append(self.asset_position)
        self.cash_balance -= order_size*asset_price*(1 + self.transaction_cost_rate*(1 if order_size > 0 else -1))
        margin_interest = sum(leverage[4] for leverage in self.leverage_maintenance_requirements)        
        portfolio_value = self.cash_balance + self.asset_position*asset_price - margin_interest
        profit = portfolio_value - self.portfolio_value
        metrics = {'time_index': self.time_index,
                   'timestamp': self.market_state_data.index[self.time_index],
                   'portfolio_value': round(portfolio_value, 2),
                   'cash_balance': round(self.cash_balance, 2),
                   'profit': round(profit, 2),
                   'asset_price': round(asset_price, 2),
                   'order_size': order_size,
                   'borrow_size': borrow_size,
                   'leverage_size': sum(leverage[3] for leverage in self.leverage_maintenance_requirements),
                   'margin_interest': round(margin_interest,3),
                   'asset_position': self.asset_position
                   }
        if self.reward_option == 0:
            reward = profit
        elif self.reward_option == 1: # encourage exploration and action
            num_consecutive_inaction = self.consecutive_inaction()
            order_inbalance = self.order_inbalance()
            reward = profit + 1e-4*np.sqrt(self.time_index)*abs(order_size) - 1e-1*num_consecutive_inaction*(num_consecutive_inaction > 30) - 1e-1*abs(np.mean(self.recent_asset_positions)) \
                - 1e-1*order_inbalance*(order_inbalance > 500) \
                - 1e-5*abs(self.portfolio_value/2 - self.cash_balance)
        elif self.reward_option == 2: # encourage exploration, penalize consecutive inaction, penalize persistent large position, penalize large leverage size, penalize by the difference between portfolio value and cash balance
            num_consecutive_inaction = self.consecutive_inaction()
            reward = profit + 1e-6*np.sqrt(self.time_index)*abs(order_size) - 1e-1*num_consecutive_inaction*(num_consecutive_inaction > 30) - 1e-1*abs(np.mean(self.recent_asset_positions)) \
                - 1e-5*abs(self.portfolio_value/2 - self.cash_balance) \
                - 1e-2*abs(sum(leverage[3] for leverage in self.leverage_maintenance_requirements))
        if np.isnan(reward):
            print('NaN reward:', reward)
        self.portfolio_value = portfolio_value
        terminated = (self.time_index == self.max_episode_steps)
        truncated = (self.portfolio_value < self.initial_balance/2) or self.margin_call_liquidation \
            or (len(self.recent_order_sizes) == 350) and all(size == 0 for size in self.recent_order_sizes) # restart a new episode if portfolio value is halved, or receives a margin call, or has been inactive for a long time
        state = np.hstack((self.market_state_data.iloc[self.time_index].values,
                           self.portfolio_value*self.cash_balance_scale,
                           self.cash_balance*self.cash_balance_scale,
                           self.asset_position*self.asset_position_scale,
                           sum(leverage[3] for leverage in self.leverage_maintenance_requirements)*self.leverage_size_scale)) # state is market state combined with portfolio state
        return (state, reward, terminated, truncated, metrics)

In [141]:
# display('Default PPO Configs:', dict(sorted(get_trainable_cls('PPO').get_default_config().to_dict().items())))
# display('Default SAC Configs:', dict(sorted(get_trainable_cls('SAC').get_default_config().to_dict().items())))

def train(experiment_config):
    model_name = (
        f"{experiment_config['algorithm']}_"
        f"{experiment_config['training_iteration']}_train_iter_"
        f"{experiment_config['train_batch_size']}_batch_size_"
        f"{experiment_config['train_market_state_data'].shape[1]-5}_tech_indicators_"
        f"{experiment_config['reward_option']}_reward_option_"
        f"{experiment_config['allow_leverage']}_leverage_option"
    )
    # remove existing .csv and .log files
    for file_path in glob.glob(f'logs/{model_name}_*.csv') + glob.glob(f'logs/{model_name}_*.log'):
        if os.path.exists(file_path):
            os.remove(file_path)

    # remove existing model checkpoint
    model_checkpoint_path = f'./checkpoints/{model_name}'
    if os.path.exists(model_checkpoint_path):
        shutil.rmtree(model_checkpoint_path)
    os.makedirs(model_checkpoint_path, exist_ok=True)

    param_space = {
        'callbacks': CustomCallbacks,
        'env': CustomEnv,
        'env_config': {
            'initial_balance': experiment_config['initial_balance'], 
            'market_state_data': experiment_config['train_market_state_data'], 
            'reward_option': experiment_config['reward_option'],
            'allow_leverage': experiment_config['allow_leverage'],
            'model_name': model_name
        },
        'lr_schedule': experiment_config['lr_schedule'],
        'gamma': 0.99999990717, # convert a 5% annual inflation rate to per-minute discounting rate: 1 - ((1 + 0.05)**(1/60/24/365) - 1)
        'num_cpus_per_worker': 1,
        'num_envs_per_worker': 1,
        'num_gpus': 0, # set to > 0 if GPUs are available
        'num_workers': 1, # number of remote/parallel workers to collect experiences in addition to the local worker; train_batch_size must be divisible by num_workers; length of training data must be divisible by num_workers
        'seed': 2, # set a seed for reproducibility
        'train_batch_size': experiment_config['train_batch_size'],
    }
    
    if experiment_config['algorithm'] == 'PPO':
        param_space['model'] = {
            'use_lstm': True, # use LSTM network as environment has sequential observations
            'lstm_cell_size': 128,
            'max_seq_len': 200,
            'lstm_use_prev_action': True,
            'lstm_use_prev_reward': True,
            'use_attention': True, # use attention mechanism to focus on different parts of the input sequence'
            'attention_num_transformer_units': 1,
            'attention_dim': 64,
            'attention_num_heads': 16,
            'attention_head_dim': 64,
            'attention_memory_inference': 350,
            'attention_memory_training': 3500,
            'attention_position_wise_mlp_dim': 64,
            'attention_use_n_prev_actions': 80,
            'attention_use_n_prev_rewards': 3500,
        }

    if experiment_config['algorithm'] == 'SAC':
        param_space['replay_buffer_config'] = {
            '_enable_replay_buffer_api': True,
            'type': 'MultiAgentPrioritizedReplayBuffer',
            'capacity': int(1e6),
            'prioritized_replay': False,
            'prioritized_replay_alpha': 0.6,
            'prioritized_replay_beta': 0.4,
            'prioritized_replay_eps': 1e-06,
            'worker_side_prioritization': False
        }
        param_space['optimization'] = {
            "actor_learning_rate": tune.choice([1e-3, 1e-4, 1e-5]),
            "critic_learning_rate": tune.choice([1e-3, 1e-4, 1e-5]),
            "entropy_learning_rate": tune.choice([1e-3, 1e-4, 1e-5]),
        }

    # https://docs.ray.io/en/latest/tune/tutorials/tune-stopping.html
    stop = {
        'training_iteration': experiment_config['training_iteration'],
        # 'timesteps_total': experiment_config['train_market_state_data'].shape[0]*3, # stop after going through the training data multiple times (e.g. 3 times)
    }

    tuner = tune.Tuner(
        experiment_config['algorithm'],
        param_space=param_space,
        run_config=air.RunConfig(
            stop=stop,
            storage_path=os.path.join(os.getcwd(), model_checkpoint_path), # if not specified, model checkpoint would be saved at '~/ray_results'
            checkpoint_config=air.CheckpointConfig(
                num_to_keep=1,
                checkpoint_score_attribute='episode_reward_mean', 
            ))
        )
    results = tuner.fit()
    return results

In [142]:
# SingleAgentRLModuleSpec(module_class=<class 'ray.rllib.algorithms.ppo.torch.ppo_torch_rl_module.PPOTorchRLModule'>, observation_space=None, action_space=None, model_config_dict=None, catalog_class=<class 'ray.rllib.algorithms.ppo.ppo_catalog.PPOCatalog'>, load_state_path=None)
# algorithm_restored = Algorithm.from_checkpoint(results.get_best_result().checkpoint)

In [143]:
# https://docs.ray.io/en/latest/rllib/rllib-saving-and-loading-algos-and-policies.html#how-do-i-restore-an-algorithm-from-a-checkpoint

def find_checkpoint_dir(root_dir):
    for root, dirs, _ in os.walk(root_dir):
        for dir in dirs:
            if dir.startswith("checkpoint"):
                return os.path.join(root, dir)
    return None

def inference(experiment_config):
    if not os.path.exists('./inferences'):
        os.makedirs('./inferences', exist_ok=True)

    model_name = (
        f"{experiment_config['algorithm']}_"
        f"{experiment_config['training_iteration']}_train_iter_"
        f"{experiment_config['train_batch_size']}_batch_size_"
        f"{experiment_config['train_market_state_data'].shape[1]-5}_tech_indicators_"
        f"{experiment_config['reward_option']}_reward_option_"
        f"{experiment_config['allow_leverage']}_leverage_option"
    )

    inference_file = open(os.path.join('inferences', model_name) + '.csv', 'w', newline='')
    writer = csv.writer(inference_file)
    header = ['time_index', 'timestamp', 'portfolio_value', 'cash_balance', 'profit', 'asset_price', 'order_size', 'borrow_size', 'leverage_size', 'margin_interest', 'asset_position']
    writer.writerow(header)

    print('Loading model checkpoint:', find_checkpoint_dir(os.path.join('checkpoints', model_name)))
    algorithm_restored = Algorithm.from_checkpoint(find_checkpoint_dir(os.path.join('checkpoints', model_name)))
    
    os.makedirs('./model', exist_ok=True)
    algorithm_restored.export_policy_model('./model')

    state = algorithm_restored.get_policy().model.get_initial_state()
    init_prev_a = prev_a = None
    init_prev_r = prev_r = None
    if experiment_config['algorithm'] == 'PPO':
        model_config = algorithm_restored.get_config().to_dict()['model']
        attention_use_n_prev_actions = model_config['attention_use_n_prev_actions']
        attention_use_n_prev_rewards = model_config['attention_use_n_prev_rewards']
        if attention_use_n_prev_actions:
            init_prev_a = prev_a = np.array([0]*attention_use_n_prev_actions)
        if attention_use_n_prev_rewards:
            init_prev_r = prev_r = np.array([0.0]*attention_use_n_prev_rewards)

    test_env = CustomEnv(env_config={
        'initial_balance': experiment_config['initial_balance'], 
        'market_state_data': experiment_config['test_market_state_data'], 
        'reward_option': experiment_config['reward_option'],
        'allow_leverage': experiment_config['allow_leverage'],
        'model_name': model_name
        })
    observation, _ = test_env.reset()

    while True:
        if experiment_config['algorithm'] == 'PPO':
            action, state_out, _ = algorithm_restored.compute_single_action(
                observation=observation,
                state=state,
                prev_action=prev_a,
                prev_reward=prev_r,
                explore=False,
                policy_id='default_policy',
            )
        else:
            action = algorithm_restored.compute_single_action(observation)
            
        observation, reward, terminated, truncated, metrics = test_env.step(action)
        writer.writerow(metrics.values())
        
        if test_env.time_index % 10000 == 0:
            print('time_index:', test_env.time_index)
            print(f"raw action: {action[0]:.2f}; order size: {metrics['order_size']}")
        if truncated:
            if test_env.time_index % 10000 == 0:
                print(f'truncated after time index: {test_env.time_index}!')
                print(f"raw action: {action[0]:.2f}; order size: {metrics['order_size']}")
        if terminated:
            print(f'terminated after time index: {test_env.time_index}!')
            break
        else:
            if experiment_config['algorithm'] == 'PPO':
                state = state_out
            if init_prev_a is not None:
                prev_a = action
            if init_prev_r is not None:
                prev_r = reward

    inference_file.close()

In [144]:
ray.shutdown()
ray.init()

experiment_config = {
    'algorithm': 'PPO',
    # 'algorithm': 'SAC',
    'initial_balance': 1e6,
    'train_market_state_data': train_market_state_data,
    # 'train_market_state_data': train_market_state_data[train_market_state_data.columns[:5]],
    'test_market_state_data': test_market_state_data,
    # 'test_market_state_data': test_market_state_data[test_market_state_data.columns[:5]],
    'allow_leverage': True,
    # 'allow_leverage': False,
    # 'reward_option': 0, 
    # 'reward_option': 1,
    'reward_option': 2,
    'lr_schedule': [
        (0, 0.0005), # (timestep, learning_rate)
        (500000, 0.00025), # halving learning rate after passing almost the entrie training data
        (1000000, 0.000125),
        (20000000, 0.0000625)
    ],
    'train_batch_size': 1000,
    'training_iteration': 100, # note when training_iteration is set to 500, it corresponds to 500*1000 timesteps ~ the length of training data (508536)
}

results = train(experiment_config)
display(results.get_best_result('episode_reward_mean', 'max', 'avg').metrics)

(pid=92464) 2023-12-14 13:42:29,866	WARNING __init__.py:10 -- PG has/have been moved to `rllib_contrib` and will no longer be maintained by the RLlib team. You can still use it/them normally inside RLlib util Ray 2.8, but from Ray 2.9 on, all `rllib_contrib` algorithms will no longer be part of the core repo, and will therefore have to be installed separately with pinned dependencies for e.g. ray[rllib] and other packages! See https://github.com/ray-project/ray/tree/master/rllib_contrib#rllib-contrib for more information on the RLlib contrib effort.
(RolloutWorker pid=92529) 2023-12-14 13:42:34,595	WARNING env.py:162 -- Your env doesn't have a .spec.max_episode_steps attribute. Your horizon will default to infinity, and your environment will not be reset.
(raylet) Spilled 2446 MiB, 70 objects, write throughput 1548 MiB/s. Set RAY_verbose_spill_logs=0 to disable this message.
(PPO pid=92464) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/Users/guo/Library/CloudSt

{'custom_metrics': {},
 'episode_media': {},
 'info': {'learner': {'__all__': {'num_agent_steps_trained': 12800.0,
    'num_env_steps_trained': 1000.0,
    'total_loss': 9.98841126759847},
   'default_policy': {'total_loss': 9.98841126759847,
    'policy_loss': -0.002409553485146413,
    'vf_loss': 9.99082056681315,
    'vf_loss_unclipped': 23963797.333333332,
    'vf_explained_var': 2.410014470418294e-05,
    'entropy': 1.0652398268381755,
    'mean_kl_loss': 0.0003759171658354414,
    'default_optimizer_lr': 5e-05,
    'curr_lr': 5e-05,
    'curr_entropy_coeff': 0.0,
    'curr_kl_coeff': 1.5777218339519106e-31}},
  'num_env_steps_sampled': 100000,
  'num_env_steps_trained': 0,
  'num_agent_steps_sampled': 100000,
  'num_agent_steps_trained': 0},
 'sampler_results': {'episode_reward_max': nan,
  'episode_reward_min': nan,
  'episode_reward_mean': nan,
  'episode_len_mean': nan,
  'episode_media': {},
  'episodes_this_iter': 0,
  'policy_reward_min': {},
  'policy_reward_max': {},
  'p

In [145]:
inference(experiment_config)

Loading model checkpoint: checkpoints/PPO_100_train_iter_1000_batch_size_11_tech_indicators_2_reward_option_True_leverage_option/PPO_2023-12-14_13-42-26/PPO_CustomEnv_e8d60_00000_0_2023-12-14_13-42-26/checkpoint_000000


(RolloutWorker pid=95238) 2023-12-14 13:44:37,109	WARNING __init__.py:10 -- PG has/have been moved to `rllib_contrib` and will no longer be maintained by the RLlib team. You can still use it/them normally inside RLlib util Ray 2.8, but from Ray 2.9 on, all `rllib_contrib` algorithms will no longer be part of the core repo, and will therefore have to be installed separately with pinned dependencies for e.g. ray[rllib] and other packages! See https://github.com/ray-project/ray/tree/master/rllib_contrib#rllib-contrib for more information on the RLlib contrib effort.
(RolloutWorker pid=95238) 2023-12-14 13:44:37,128	WARNING env.py:162 -- Your env doesn't have a .spec.max_episode_steps attribute. Your horizon will default to infinity, and your environment will not be reset.


time_index: 10000
raw action: -1.24; order size: 0
time_index: 20000
raw action: 0.98; order size: 0
time_index: 30000
raw action: -0.08; order size: 0
time_index: 40000
raw action: 3.06; order size: 0
time_index: 50000
raw action: 0.15; order size: 0
truncated after time index: 50000!
raw action: 0.15; order size: 0
time_index: 60000
raw action: 2.20; order size: 0
time_index: 70000
raw action: 3.93; order size: 0
time_index: 80000
raw action: 2.65; order size: 0
time_index: 90000
raw action: 0.94; order size: 0
time_index: 100000
raw action: 0.68; order size: 0
truncated after time index: 100000!
raw action: 0.68; order size: 0
time_index: 110000
raw action: 2.29; order size: 0
time_index: 120000
raw action: 1.25; order size: 0
terminated after time index: 127134!


In [146]:
import torch

model = torch.load('./model/model.pt')
model.eval()
model

PPOTorchRLModule(
  (encoder): TorchStatefulActorCriticEncoder(
    (actor_encoder): TorchLSTMEncoder(
      (tokenizer): TorchMLPEncoder(
        (net): TorchMLP(
          (mlp): Sequential(
            (0): Linear(in_features=20, out_features=256, bias=True)
            (1): Tanh()
            (2): Linear(in_features=256, out_features=256, bias=True)
            (3): Tanh()
          )
        )
      )
      (lstm): LSTM(256, 128, batch_first=True)
    )
    (critic_encoder): TorchLSTMEncoder(
      (tokenizer): TorchMLPEncoder(
        (net): TorchMLP(
          (mlp): Sequential(
            (0): Linear(in_features=20, out_features=256, bias=True)
            (1): Tanh()
            (2): Linear(in_features=256, out_features=256, bias=True)
            (3): Tanh()
          )
        )
      )
      (lstm): LSTM(256, 128, batch_first=True)
    )
  )
  (pi): TorchMLPHead(
    (net): TorchMLP(
      (mlp): Sequential(
        (0): Linear(in_features=128, out_features=2, bias=True)
 